# Compare retrieval experiments with evidence and abstention

Build two deterministic answer systems, evaluate them on a fixed, authored synthetic question set, and compare correctness, evidence, and abstention. This worked regression suite is not an independently held-out benchmark.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** compare deterministic retrieval with and without abstention; score answer correctness and source grounding independently; expose evaluator false positives and interpret small-sample uncertainty.

**Prerequisites:** Understand Python collections and basic proportions. The local experiment uses only the standard library. Recording an optional Phoenix experiment additionally requires its audited client profile, an authorized HTTPS endpoint, and a separately uploaded synthetic dataset.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.

For optional Phoenix integration, install the [audited optional service profile](../../_requirements/optional-services.lock.txt) into the same isolated environment. HTTPX is included in the base environment. An installed package does not imply a live service has been verified.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Define evidence and independent evaluation cases

The corpus describes a fictional makerspace. Answers and source identifiers are independently specified in the evaluation set. Unsupported questions have no expected source; a safe system must abstain.


In [ ]:
import json
import math
import re
from collections import Counter

CORPUS = {
    "p1": "Equipment loans last 14 days. Equipment includes cameras and microphones.",
    "p2": "Laser cutter induction lasts 45 minutes. Safety goggles are mandatory.",
    "p3": "The ceramics studio opens at 09:00 on Saturday and closes at 16:00.",
    "p4": "Workshop cancellations qualify for a refund until 48 hours before the workshop.",
}
CASES = [
    ("How many days can I borrow equipment?", "14", "p1", "direct"),
    ("How long are camera loans?", "14", "p1", "paraphrase"),
    ("How many minutes does laser induction take?", "45", "p2", "direct"),
    ("What must I wear for laser safety?", "goggles", "p2", "paraphrase"),
    ("When does ceramics open Saturday?", "09:00", "p3", "direct"),
    ("What is the Saturday ceramics closing time?", "16:00", "p3", "paraphrase"),
    ("How early must I cancel a workshop for a refund?", "48", "p4", "direct"),
    ("What is the membership price?", None, None, "unsupported"),
    ("Is parking free?", None, None, "unsupported"),
    ("Can I bring a pet?", None, None, "unsupported"),
]
assert len({q for q, *_ in CASES}) == len(CASES)
print(
    {
        "documents": len(CORPUS),
        "cases": len(CASES),
        "slices": dict(Counter(x[3] for x in CASES)),
    }
)

## 2. Implement retrieval without accessing the answer key

The baseline always returns a document, including when no evidence matches. The candidate uses stop-word filtering, a small declared vocabulary normalization, IDF-weighted overlap, and an abstention threshold. It returns evidence verbatim; there is no generated model answer.


In [ ]:
STOP = {
    "the",
    "a",
    "an",
    "is",
    "i",
    "how",
    "what",
    "for",
    "does",
    "can",
    "at",
    "to",
    "must",
    "my",
    "of",
    "in",
    "when",
    "many",
}
NORMALIZE = {
    "borrow": "loans",
    "loan": "loans",
    "camera": "cameras",
    "cancel": "cancellations",
    "cancellation": "cancellations",
    "closing": "closes",
    "open": "opens",
}


def tokens(text):
    return [NORMALIZE.get(t, t) for t in re.findall(r"[a-z]+", text.lower()) if t not in STOP]


DOCS = {k: set(tokens(v)) for k, v in CORPUS.items()}
IDF = {
    term: math.log((1 + len(DOCS)) / (1 + sum(term in ts for ts in DOCS.values()))) + 1
    for ts in DOCS.values()
    for term in ts
}


def answer(question, abstain=False):
    query = set(tokens(question))
    scored = sorted(
        ((sum(IDF[t] for t in query & terms), key) for key, terms in DOCS.items()),
        reverse=True,
    )
    score, key = scored[0]
    if abstain and score < 1.5:
        return {"answer": "INSUFFICIENT_EVIDENCE", "source": None, "score": score}
    return {"answer": CORPUS[key], "source": key, "score": round(score, 3)}


assert answer("membership price", True)["source"] is None
assert answer("camera loans", True)["source"] == "p1"
print(answer(CASES[0][0], True))

## 3. Evaluate three separate properties

Correctness requires the independently specified fact and correct source. Grounding measures whether the answer is copied from its cited document. Unsupported questions require both a missing source and the exact `INSUFFICIENT_EVIDENCE` sentinel. A missing citation with fabricated answer text must fail correctness and grounding. A response may be grounded yet irrelevant.


In [ ]:
def score_output(output, fact, source):
    if (
        not isinstance(output, dict)
        or not {"answer", "source"} <= output.keys()
        or not isinstance(output["answer"], str)
        or (output["source"] is not None and not isinstance(output["source"], str))
    ):
        raise ValueError("Invalid answer/source contract")
    # A missing citation alone does not make fabricated text an abstention.
    abstained = output["source"] is None and output["answer"] == "INSUFFICIENT_EVIDENCE"
    correct = (
        abstained if source is None else (output["source"] == source and fact in output["answer"])
    )
    grounded = abstained or (
        output["source"] is not None and output["answer"] == CORPUS.get(output["source"])
    )
    return {
        "correct": int(correct),
        "grounded": int(grounded),
        "abstained": int(abstained),
    }


def evaluate(system):
    return [
        {
            "question": question,
            "slice": slice_name,
            **score_output(system(question), fact, source),
        }
        for question, fact, source, slice_name in CASES
    ]


baseline = evaluate(lambda q: answer(q, False))
candidate = evaluate(lambda q: answer(q, True))


def summarize(rows):
    return {
        s: {
            "n": sum(r["slice"] == s for r in rows),
            "accuracy": sum(r["correct"] for r in rows if r["slice"] == s)
            / sum(r["slice"] == s for r in rows),
        }
        for s in sorted({r["slice"] for r in rows})
    }


print(json.dumps({"baseline": summarize(baseline), "candidate": summarize(candidate)}, indent=2))
assert sum(r["correct"] for r in candidate) > sum(r["correct"] for r in baseline)
assert all(r["correct"] for r in candidate if r["slice"] == "unsupported")

## 4. Illustrate a binomial interval and test the evaluator

The Wilson calculation below illustrates a binomial interval for independent Bernoulli trials from a defined task population. These deliberately authored cases are not such a sample, so this interval does not estimate deployment accuracy or generalization uncertainty. A deliberately wrong response must fail; otherwise an evaluator can appear perfect by construction.


In [ ]:
def wilson(successes, n, z=1.96):
    p = successes / n
    den = 1 + z * z / n
    center = (p + z * z / (2 * n)) / den
    radius = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (round(center - radius, 3), round(center + radius, 3))


correct = sum(r["correct"] for r in candidate)
paired = [b["correct"] - a["correct"] for a, b in zip(baseline, candidate)]
assert sum(paired) == correct - sum(r["correct"] for r in baseline)
mutant = evaluate(lambda q: {"answer": "A fictional wrong value", "source": "p1"})
assert sum(r["correct"] for r in mutant) == 0
uncited_fabrication = evaluate(
    lambda q: {"answer": "Membership costs 999 dollars and parking is free.", "source": None}
)
assert all(
    not any(row[key] for key in ("correct", "grounded", "abstained")) for row in uncited_fabrication
)
assert score_output({"answer": "INSUFFICIENT_EVIDENCE", "source": "p1"}, None, None) == {
    "correct": 0,
    "grounded": 0,
    "abstained": 0,
}
for malformed in [None, {}, {"answer": [], "source": None}, {"answer": "x", "source": []}]:
    try:
        score_output(malformed, None, None)
    except ValueError:
        pass
    else:
        raise AssertionError("Malformed answer contract was accepted")
print(
    {
        "accuracy": correct / len(CASES),
        "wilson_95": wilson(correct, len(CASES)),
        "paired_improvements": sum(x > 0 for x in paired),
        "regressions": sum(x < 0 for x in paired),
    }
)

## 5. Optional: record the same experiment in Phoenix

Install the [audited optional service profile](../../_requirements/optional-services.lock.txt), set `PHOENIX_API_KEY` and `PHOENIX_COLLECTOR_ENDPOINT` through your process environment, and provide `PHOENIX_DATASET`. In a disposable Phoenix instance, upload exactly the ten authored `CASES` above: each example's `input` is `{"question": question}` and its `output` is `{"fact": fact, "source": source}`. Use JSON null for the unsupported cases' fact and source, and an empty metadata object for every example. Phoenix binds an evaluator's `expected` argument to the dataset's `output` field, not a separate field named `expected`.

The adapter verifies the fetched dataset before creating an experiment. An injected Phoenix HTTPX client must own its base URL and Authorization header; constructor credentials are ignored when that client is supplied. TLS, redirects, ambient proxies, and timeouts are explicit. These single-write methods do not retry automatically. This example uses the public `create`, `log_run`, `log_evaluation`, and `get_experiment` methods to record all ten local results explicitly. These methods use the supplied HTTP client and do not invoke trace export. The batch `run_experiment` wrapper additionally creates an OTLP transport and can return partial results; it is deliberately avoided here.

Phoenix can return incomplete results or failed-to-persist evaluations without raising an exception. The checks below require every expected task and evaluation once, no errors, and acknowledged server identifiers for each write. `get_experiment` reconstructs evaluation identifiers on the client; those reconstructed values are not storage receipts. Persistence is checked through fetched annotation contents and their complete association with the expected task runs, evaluator name, and score. HTTP and schema failures produce only a sanitized notebook error, because provider error bodies can contain private values. Successful writes are acknowledged by server identifiers. Each fetched task output must exactly match the deterministic output actually submitted for that run, and every linked evaluation must have the expected score without an error; matching a fact substring cannot establish this integrity check. This does not automatically export a local file or verify a hosted model.


In [ ]:
import os
from datetime import datetime, timezone
from urllib.parse import urlsplit


def verify_phoenix_dataset(dataset):
    expected = {question: (fact, source) for question, fact, source, _ in CASES}
    examples = dataset.examples
    seen_questions, seen_ids = set(), set()
    if len(examples) != len(expected):
        raise ValueError("Phoenix dataset must contain exactly the authored synthetic cases")
    for example in examples:
        question = example.get("input", {}).get("question")
        identifier = example.get("node_id") or example.get("id")
        reference = example.get("output", {})
        if (
            question not in expected
            or question in seen_questions
            or not isinstance(identifier, str)
            or not identifier
            or identifier in seen_ids
            or reference != dict(zip(("fact", "source"), expected[question]))
            or example.get("input") != {"question": question}
            or example.get("metadata") != {}
        ):
            raise ValueError("Phoenix dataset differs from the authored synthetic cases")
        seen_questions.add(question)
        seen_ids.add(identifier)
    return {example.get("node_id") or example["id"]: example for example in examples}


def verify_phoenix_experiment(experiment, examples, acknowledged_outputs):
    def persisted(identifier):
        return (
            isinstance(identifier, str)
            and bool(identifier)
            and not identifier.startswith(("DRY_RUN", "temp-"))
        )

    runs = experiment.get("task_runs", [])
    evaluations = experiment.get("evaluation_runs", [])
    if len(runs) != len(examples) or len(evaluations) != len(examples):
        raise ValueError("Phoenix experiment has incomplete task or evaluation coverage")
    seen_examples, run_ids = set(), set()
    for run in runs:
        example_id, run_id = run.get("dataset_example_id"), run.get("id")
        if (
            example_id not in examples
            or example_id in seen_examples
            or not persisted(run_id)
            or run_id in run_ids
            or run_id not in acknowledged_outputs
            or run.get("repetition_number") != 1
            or run.get("error") is not None
        ):
            raise ValueError("Phoenix task coverage contains invalid or failed runs")
        if run.get("output") != acknowledged_outputs[run_id]:
            raise ValueError("Phoenix read-back task content differs from the acknowledged write")
        reference = examples[example_id]["output"]
        if not score_output(run.get("output"), reference["fact"], reference["source"])["correct"]:
            raise ValueError("Phoenix task output differs from the expected synthetic result")
        seen_examples.add(example_id)
        run_ids.add(run_id)
    seen_runs = set()
    for evaluation in evaluations:
        if (
            evaluation.experiment_run_id not in run_ids
            or evaluation.experiment_run_id in seen_runs
            or evaluation.error is not None
            or evaluation.name != "exact_fact"
            or not isinstance(evaluation.result, dict)
            or type(evaluation.result.get("score")) not in (int, float)
            or evaluation.result.get("score") != 1.0
        ):
            raise ValueError("Phoenix evaluations are incomplete, failed, or not persisted")
        seen_runs.add(evaluation.experiment_run_id)
    return {"task_runs": len(runs), "persisted_evaluations": len(evaluations)}


RUN_LIVE = False
if RUN_LIVE:
    import httpx

    endpoint = os.environ["PHOENIX_COLLECTOR_ENDPOINT"]
    try:
        parsed = urlsplit(endpoint)
    except ValueError:
        # URL parsing can echo credentials from malformed Unicode authorities.
        raise ValueError("Use an HTTPS Phoenix endpoint without embedded credentials") from None
    if (
        parsed.scheme != "https"
        or not parsed.hostname
        or parsed.query
        or parsed.fragment
        or parsed.username
        or parsed.password
    ):
        raise ValueError("Use an HTTPS Phoenix endpoint without embedded credentials")

    def live_task(input):
        return answer(input["question"], True)

    def exact_fact(output, expected):
        return bool(score_output(output, expected["fact"], expected["source"])["correct"])

    try:
        from phoenix.client import Client

        with httpx.Client(
            base_url=endpoint,
            headers={"Authorization": "Bearer " + os.environ["PHOENIX_API_KEY"]},
            timeout=httpx.Timeout(60.0, connect=10.0),
            follow_redirects=False,
            trust_env=False,
        ) as transport:
            client = Client(http_client=transport)
            dataset = client.datasets.get_dataset(dataset=os.environ["PHOENIX_DATASET"], timeout=60)
            verified_examples = verify_phoenix_dataset(dataset)
            created = client.experiments.create(
                dataset_id=dataset.id,
                dataset_version_id=dataset.version_id,
                experiment_name="synthetic-evidence-abstention",
                repetitions=1,
                timeout=60,
            )
            experiment_id = created["id"]
            if not isinstance(experiment_id, str) or not experiment_id:
                raise ValueError("Phoenix did not acknowledge an experiment identifier")
            acknowledged_runs, acknowledged_evaluations = set(), set()
            acknowledged_outputs = {}
            for example_id, example in verified_examples.items():
                started = datetime.now(timezone.utc)
                output = live_task(example["input"])
                run = client.experiments.log_run(
                    experiment_id=experiment_id,
                    dataset_example_id=example_id,
                    output=output,
                    start_time=started,
                    end_time=datetime.now(timezone.utc),
                    repetition_number=1,
                    timeout=60,
                )
                run_id = run["id"]
                if not isinstance(run_id, str) or not run_id or run_id in acknowledged_runs:
                    raise ValueError("Phoenix did not acknowledge a unique task identifier")
                evaluation = client.experiments.log_evaluation(
                    experiment_run_id=run_id,
                    name="exact_fact",
                    score=float(exact_fact(output, example["output"])),
                    annotator_kind="CODE",
                    timeout=60,
                )
                evaluation_id = evaluation["id"]
                if (
                    not isinstance(evaluation_id, str)
                    or not evaluation_id
                    or evaluation_id in acknowledged_evaluations
                ):
                    raise ValueError("Phoenix did not acknowledge a unique evaluation identifier")
                acknowledged_outputs[run_id] = dict(output)
                acknowledged_runs.add(run_id)
                acknowledged_evaluations.add(evaluation_id)
            # Read-back evaluation IDs are generated by this SDK; inspect persisted contents,
            # not those reconstructed IDs, against the acknowledged writes.
            experiment = client.experiments.get_experiment(experiment_id=experiment_id)
            recorded = verify_phoenix_experiment(
                experiment, verified_examples, acknowledged_outputs
            )
            if {run["id"] for run in experiment["task_runs"]} != acknowledged_runs:
                raise ValueError("Phoenix read-back identifiers differ from acknowledged writes")
    except Exception:
        raise RuntimeError(
            "Phoenix experiment failed; no complete persisted result was verified"
        ) from None
    print("Verified synthetic Phoenix experiment:", recorded)
else:
    print("Phoenix integration not run; all preceding metrics are local.")

## Expected local results

The baseline answers 7/10 cases correctly. The candidate answers 10/10, with three improvements and no regressions on this authored set. The Wilson formula returns approximately [0.722, 1.000] for ten successes, as a calculation exercise. Since these cases were deliberately authored rather than sampled independently from a defined task population, that interval does not estimate deployment accuracy or generalization uncertainty.

**Try next:** author unsupported questions that share policy vocabulary and document the system's failures before tuning. For generalization estimates, design a sampling protocol and independently label a separate evaluation set before inspecting its results.


## Interpretation, limitations, and cleanup

The candidate improves refusal behavior on this authored set, not proven generalization. Exact substring facts are inadequate for free-form reasoning. Expand held-out topics, paraphrases, contradictory evidence, and independent human review before adding a model-based judge. Grounding alone cannot certify correctness.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [Phoenix experiment client reference](https://arize-phoenix.readthedocs.io/projects/client/api/experiments.html)
